# 04 The neural tangent kernel and lazy training

Lecture 10, training and generalization. Book chapter 8 (the neural tangent
kernel and lazy training).

**Goals**

- Compute the empirical neural tangent kernel (NTK)
  $K = J(X) J(X)^\top$ with `glassnn.analysis.empirical_ntk`.
- See that, in the NTK parametrization, the kernel at initialization
  becomes deterministic as the width grows (Jacot et al. 2018).
- Train networks of increasing width and measure how much the kernel and
  the parameters move; compare the trained network with its linearization
  (Lee et al. 2019; Chizat et al. 2019).

**Expected finding** (asserted in `tests/test_examples.py::test_04_*`): the
relative change of the NTK during training decreases with the width
(roughly like $1/\sqrt{\text{width}}$), and so does the relative change of
the parameters, while the training error still decreases.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import glassnn.functional as F
from glassnn import Tensor, backend, manual_seed, nn, no_grad, optim
from glassnn.analysis import empirical_ntk

backend.set_default_dtype("float64")

# 20 training points on a curve; the second input is a constant.
rng = np.random.default_rng(0)
X = np.hstack([rng.uniform(-1, 1, size=(20, 1)), np.full((20, 1), 0.5)])
y = np.sin(3 * X[:, :1])
X_grid = np.hstack([np.linspace(-1, 1, 101)[:, None], np.full((101, 1), 0.5)])


def mlp(width):
    # One hidden layer, NTK parametrization: weights N(0, 1), scaled by 1/sqrt(fan_in).
    return nn.Sequential(
        nn.Linear(2, width, parametrization="ntk"),
        nn.ReLU(),
        nn.Linear(width, 1, parametrization="ntk"),
    )

## The NTK at initialization concentrates

For each width, ten networks with different seeds: the spread of
$K(x_1, x_2)$ over seeds shrinks as the width grows.

In [ ]:
widths = [10, 100, 1000, 10000]
fig, ax = plt.subplots(figsize=(6, 3.5))
for width in widths:
    values = []
    for seed in range(10):
        manual_seed(seed)
        values.append(empirical_ntk(mlp(width), X[:2]).data[0, 1])
    ax.plot([width] * len(values), values, "o", alpha=0.5)
    print(f"width {width:6d}: K(x1, x2) = {np.mean(values):.3f} ± {np.std(values):.3f}")
ax.set_xscale("log")
ax.set_xlabel("width")
ax.set_ylabel("$K(x_1, x_2)$ at initialization")
plt.show()

## Training: how much do the kernel and the parameters move?

Full-batch gradient descent on $L = \frac{1}{2N}\sum_n (f(x_n) - y_n)^2$
with learning rate 0.5 for 500 steps, the same for all widths.

In [ ]:
def flat_parameters(model):
    return np.concatenate([p.data.ravel() for p in model.parameters()])


def train(width, seed=0, steps=500, lr=0.5):
    manual_seed(seed)
    model = mlp(width)
    record = {
        "K0": empirical_ntk(model, X).data,
        "K_grid0": empirical_ntk(model, X_grid, X).data,
        "theta0": flat_parameters(model),
    }
    with no_grad():
        record["f0"] = model(Tensor(X)).data
        record["f_grid0"] = model(Tensor(X_grid)).data
    optimizer = optim.SGD(model.parameters(), lr=lr)
    for _ in range(steps):
        optimizer.zero_grad()
        (0.5 * F.mse_loss(model(Tensor(X)), Tensor(y))).backward()
        optimizer.step()
    record["K1"] = empirical_ntk(model, X).data
    record["theta1"] = flat_parameters(model)
    with no_grad():
        record["train_error"] = F.mse_loss(model(Tensor(X)), Tensor(y)).item()
        record["f_grid1"] = model(Tensor(X_grid)).data
    return record


runs = {width: [train(width, seed) for seed in range(3)] for width in widths}


def relative_change(a, b):
    return np.linalg.norm(b - a) / np.linalg.norm(a)


kernel_change = {
    w: np.median([relative_change(r["K0"], r["K1"]) for r in runs[w]]) for w in widths
}
parameter_change = {
    w: np.median([relative_change(r["theta0"], r["theta1"]) for r in runs[w]])
    for w in widths
}
for w in widths:
    print(
        f"width {w:6d}: kernel change {kernel_change[w]:.4f}, "
        f"parameter change {parameter_change[w]:.4f}, "
        f"train error {np.median([r['train_error'] for r in runs[w]]):.4f}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(
    widths,
    list(kernel_change.values()),
    "o-",
    label=r"$\|K_T - K_0\| / \|K_0\|$",
)
ax.loglog(
    widths,
    list(parameter_change.values()),
    "s-",
    label=r"$\|\theta_T - \theta_0\| / \|\theta_0\|$",
)
guide = kernel_change[widths[0]] * np.sqrt(widths[0] / np.array(widths))
ax.loglog(widths, guide, "k:", label=r"$\propto 1/\sqrt{\mathrm{width}}$")
ax.set_xlabel("width")
ax.set_ylabel("relative change during training")
ax.legend()
plt.show()

## The network against its linearization

If the kernel stays fixed, the outputs on the training set follow the linear
recursion $f_{t+1} = f_t - \frac{\eta}{N} K_0 (f_t - y)$, and the output at
any other point is
$f_T(x) = f_0(x) - \frac{\eta}{N} K_0(x, X) \sum_{t<T} (f_t - y)$.
This is the prediction of the *linearized* network
$f_0 + \nabla_\theta f_0^\top (\theta - \theta_0)$.

In [ ]:
def linearized_prediction(record, steps=500, lr=0.5):
    n = len(X)
    residual = record["f0"] - y
    total = np.zeros_like(residual)
    for _ in range(steps):
        total += residual
        residual = residual - lr / n * record["K0"] @ residual
    return record["f_grid0"] - lr / n * record["K_grid0"] @ total


fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, width in zip(axes, widths[1:], strict=True):
    record = runs[width][0]
    ax.plot(X_grid[:, 0], record["f_grid1"], label="trained network")
    ax.plot(X_grid[:, 0], linearized_prediction(record), "--", label="linearized")
    ax.plot(X[:, 0], y, "k.", label="training data")
    ax.set_title(f"width {width}")
axes[0].legend(fontsize=8)
plt.show()

**Finding.** In the NTK parametrization the kernel at initialization
concentrates as the width grows, and during training both the kernel and the
parameters move less and less (about $1/\sqrt{\text{width}}$), while the
training error decreases (the widest networks fit best): training is *lazy*. The trained wide network is
close to its linearization, i.e. to kernel regression with the fixed kernel
$K_0$ (Jacot et al. 2018; Lee et al. 2019). Narrow networks, and networks in
the standard parametrization with larger learning rates, leave this regime
and learn features (Chizat et al. 2019).

**Exercises.**

1. Repeat the training with `parametrization="standard"`. Which learning
   rate is stable for each width, and does the kernel still stop moving?
2. Multiply the output of the network by a factor $\alpha$ and the learning
   rate by $1/\alpha^2$ (Chizat et al. 2019). What happens for large
   $\alpha$?